In [ ]:
# pip install wikipedia-api

# Agentic Workflows

In this exercise, you will build an agentic system that generates a short research report through planning, external tool usage, and feedback integration. Your workflow will involve:

### Agents

* **Planning Agent / Writer**: Creates an outline and coordinates tasks.
* **Research Agent**: Gathers external information using tools like Arxiv, Tavily, and Wikipedia.
* **Editor Agent**: Reflects on the report and provides suggestions for improvement.

***
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS:</h4>

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment
 
---


### Research Tools

By importing `research_tools`, you gain access to several search utilities:

- `research_tools.arxiv_search_tool(query)` → search academic papers from **arXiv**  

  *Example:* `research_tools.arxiv_search_tool("neural networks for climate modeling")`

- `research_tools.tavily_search_tool(query)` → perform web searches with the **Tavily API**  

  *Example:* `research_tools.tavily_search_tool("latest trends in sunglasses fashion")`

- `research_tools.wikipedia_search_tool(query)` → retrieve summaries from **Wikipedia**  

  *Example:* `research_tools.wikipedia_search_tool("Ensemble Kalman Filter")`

Run the cell below to make them available.

In [ ]:
# =========================
# Imports
# =========================

# --- Standard library 
from datetime import datetime
import re
import json
from pydantic import BaseModel, Field
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Literal


# --- Third-party ---
from IPython.display import Markdown, display

# --- Local / project ---
import research_tools
from gates_openai import create_response, parse_response

## Exercise 1: planner_agent

### Objective
Correctly set up a call to a language model (LLM) to generate a research plan.

### Instructions

1. **Focus Areas**:
   - Ensure `parse_response` is correctly configured.
   - Pass the `model` and `input` parameters correctly:
     - **Model**: Use `"gpt-4o-mini"` by default.
     - **Messages**: Set with `{"role": "user", "content": user_prompt}`.
     - **Temperature**: Fixed at 1 for creative outputs.

### Notes

- The prompt is pre-defined and guides the LLM on task requirements.
- Only return a formatted list of steps — no extra text.

Focus on the LLM call setup to complete the task.

In [ ]:
MAX_STEPS = 4

In [ ]:
# pydantic class for planner_agent's structured output

class PlanSteps(BaseModel):
    steps: list[str] = Field(description="Clear, step-by-step research plan, where each step is a string. Each step should be atomic, executable, and must rely only on the capabilities of the given agents.", max_length=MAX_STEPS)

In [ ]:
# planner_agent

def planner_agent(topic: str, model: str = "gpt-4o-mini") -> list[str]:
    """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
        topic (str): Research topic to investigate.
        model (str): Language model to use.

    Returns:
        List[str]: A list of executable step strings.
    """

    
    # Build the user prompt
    user_prompt = f"""
    You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

    ### Available agents:
    - A research agent who can search the web, Wikipedia, and arXiv.
    - A writer agent who can draft research summaries.
    - An editor agent who can reflect and revise the drafts.

    Your job is to write a clear, step-by-step research plan, where each step is a string.
    Each step should be atomic, executable, and must rely only on the capabilities of the above agents.

    ### Rules
    - DO NOT include irrelevant tasks like "create CSV", "set up a repo", "install packages", etc.
    - DO include real research-related tasks (e.g., search, summarize, draft, revise).
    - DO assume tool use is available.
    - Maximum steps is {MAX_STEPS}
    - The final step should be to generate a Markdown document containing the complete research report.
    

    Topic: "{topic}"
    """

    ### START CODE HERE ###

    # Add the user prompt to the messages list
    messages = [{"role": "user", "content": None}]

    # Call the LLM
    response = parse_response( 
        # Pass in the model
        model=None,
        # Define the messages. Remember this is meant to be a user prompt!
        input=None,
        # Set the temperature for strict outputs
        temperature=None, 
        # Give the structured output format class
        text_format=None
    )

    ### END CODE HERE ###


    # Parse steps
    steps = response.steps

    return steps

In [ ]:
# Sanity Check
steps = planner_agent("recent news in the philippines")
for ctr, step in enumerate(steps):
    print(f"Step{ctr+1}: {step}")

## Exercise 2: research_agent

### Objective
Set up a call to a language model (LLM) to perform a research task using various tools.

### Instructions

**Focus Areas**:

- **Creating a Custom Prompt**:
  - **Define the Role**: Clearly specify the role, such as "research assistant."
  - **List Available Tools** (as strings inside the prompt, not the actual functions):
    - Use `arxiv_tool` to find academic papers.
    - Use `tavily_tool` for general web searches.
    - Use `wikipedia_tool` for accessing encyclopedic knowledge.
  - **Specify the Task**: Include a placeholder in your prompt for defining the specific task that needs to be accomplished.
  - **Include Date Information**: Add a placeholder for the current date or time to provide context.

- **Creating Messages Dict**:
  - Ensure the `messages` are correctly set with `{"role": "user", "content": prompt}`.

- **Creating Tools List**:
  - Create a list of tools for use, such as `research_tools.arxiv_search_tool`, `research_tools.tavily_search_tool`, and `research_tools.wikipedia_search_tool`.

- **Correctly Setting the Call to the LLM**:
  - Pass the `model`, `messages`, and `tools` parameters accurately.
  - Set `tool_choice` to `"auto"` for automatic tool selection.
  - Limit interactions with `max_turns=6`.

### Notes

- The function provides pre-coded blocks where you need to replace placeholder values.
- The approach allows the LLM to use tools dynamically based on the task.

Focus on accurately setting the messages, tools, and LLM call parameters to complete the task.

In [ ]:
#helper function for parallel calls
def execute_tool(call):
    try:
        args = json.loads(call.arguments)

        result = research_tools.tool_mapping[call.name](**args)
        print(f"Result for {call.name}:\n{result}")

        return {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": "\n".join(map(str, result))
        }

    except Exception as e:
        return {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": f"ERROR: {e}"
        }

In [ ]:
# research_agent

def research_agent(task: str, model: str = "gpt-4o-mini"):
    """
    Executes a research task using tools from research_tools.
    """
    print("==================================")  
    print("Research Agent")                 
    print("==================================")

    current_time = datetime.now().strftime('%Y-%m-%d')
    
    ### START CODE HERE ###

    # Create a customizable prompt by defining the role (e.g., "research assistant"),
    # listing tools (arxiv_tool, tavily_tool, wikipedia_tool) for various searches,
    # specifying the task with a placeholder, and including a current_time placeholder.
    prompt = None
    
    # Create the messages dict to pass to the LLM. Remember this is a user prompt!
    messages = None

    # Save all of your available tools in the tools list. These can be found in the research_tools module.
    # You can identify each tool in your list like this: 
    # research_tools.<tool_definition>, where <tool_definition> is replaced with the function definition.
    tools = None
    
    # Call the model with tools enabled
    response = create_response(  
        # Set the model
        model=None,
        # Pass in the messages. You already defined this!
        input=None,
        # Pass in the tools list. You already defined this!
        tools=None,
        tool_choice = 'required'
    )  
    
    ### END CODE HERE ###

    function_calls = [
        item for item in response.output
        if item.type == "function_call"
    ]

    if not function_calls:
        return response.output_text

    while True:

        tool_outputs = []

        with ThreadPoolExecutor(max_workers=5) as executor:

            futures = [
                executor.submit(execute_tool, call)
                for call in function_calls
            ]

            for future in as_completed(futures):
                tool_outputs.append(future.result())

        messages.extend(tool_outputs)

        response = create_response(
            model = model,
            input = tool_outputs,
            tools = tools,
            previous_response_id = response.id
        )


        function_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if function_calls:
            print("Function calls found...")
            continue
        else:
            return response.output_text

In [ ]:
# Sanity Check
first_response = research_agent("recent news in the Philippines")
print("\nFinal Response:\n")
print(first_response)

## Exercise 3: writer_agent

### Objective
Set up a call to a language model (LLM) for executing writing tasks like drafting, expanding, or summarizing text.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of a writing agent focused on generating academic or technical content.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The function is designed to produce well-structured text by setting the correct prompts.
- Temperature is set to 1.0 to allow for creative variance in the writing outputs.

Ensure the system prompt and messages are defined properly to achieve a structured output from the LLM.

In [ ]:
# writer_agent
def writer_agent(task: str, model: str = "gpt-4o-mini") -> str: # @REPLACE def writer_agent(task: str, model: str = None) -> str:
    """
    Executes writing tasks, such as drafting, expanding, or summarizing text.
    """
    print("==================================")
    print("Writer Agent")
    print("==================================")

    ### START CODE HERE ###
    
    # Create the system prompt.
    # This should assign the LLM the role of a writing agent specialized in generating well-structured academic or technical content
    system_prompt = None

    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = None

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = None

    # Add both system and user messages to the messages list
    messages = None
    
    ### END CODE HERE ###

    response = create_response(
        model=model, 
        input=messages,
        temperature=1.0
    )

    return response.output_text

In [ ]:
# Sanity Check
enriched_task = f"""
You are writer_agent.

Here is the context of what has been done so far:
Step 1 executed by research_agent:
{first_response}

Your next task is:
"Summarize the key findings and important events from the gathered information"
"""

second_response = writer_agent(task=enriched_task)
print(second_response)


## Exercise 4: editor_agent

### Objective
Configure a call to a language model (LLM) to perform editorial tasks such as reflecting, critiquing, or revising drafts.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of an editor agent whose task is to reflect on, critique, or improve drafts.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The editor agent is tailored for enhancing the quality of text by setting an appropriate role and task in the prompts.
- Temperature is set to 0.7, balancing creativity and coherence in editorial outputs.

Ensure the system prompt and messages are accurately set up to perform effective editorial tasks with the LLM.

In [ ]:
class ReflectionOutput(BaseModel):
    reflection: str = Field(description="The reflection of a report that should cover strengths, limitations, suggestions, and opportunities.")
    revised_report: str = Field(description="The revised report that should incorporate the reflection elements to improve clarity and academic tone.")

In [ ]:
# editor_agent
def editor_agent(task: str, model: str = "gpt-4o-mini") -> str:
    """
    Executes editorial tasks such as reflection, critique, or revision.
    """
    print("==================================")
    print("Editor Agent")
    print("==================================")
    
    ### START CODE HERE ###

    # Create the system prompt.
    # This should assign the LLM the role of an editor agent specialized in reflecting on, critiquing, or improving existing drafts.
    system_prompt = None
    
    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = None
    
    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = None
    
    # Add both system and user messages to the messages list
    messages = None
    
    ### END CODE HERE ###
    
    response = parse_response(
        model=model, 
        input=messages,
        temperature=0.7,
        text_format=ReflectionOutput 
    )
    
    print(response.reflection)
    return response.revised_report

In [ ]:
# Sanity Check
enriched_task = f"""
You are editor_agent.

Here is the context of what has been done so far:
Step 1 executed by research_agent:
{first_response}
Step 2 executed by writer_agent:
{second_response}

Your next task is:
"Revise the summary for clarity and coherence"
"""

third_response = editor_agent(task=enriched_task)
print(third_response)


In [ ]:
# Sanity Check
enriched_task = f"""
You are writer_agent.

Here is the context of what has been done so far:
Step 1 executed by research_agent:
{first_response}
Step 2 executed by writer_agent:
{second_response}
step 3 executed by editor_agent:
{third_response}


Your next task is:
"Generate a Markdown document containing the complete research report on recent news in the Philippines"
"""

fourth_response = writer_agent(task=enriched_task)
print(fourth_response)


### The Executor Agent

The `executor_agent` manages the workflow by executing each step of a given plan. It:

1. Decides **which agent** (`research_agent`, `writer_agent`, or `editor_agent`) should handle the step.
2. Builds context from the outputs of previous steps.
3. Sends the enriched task to the selected agent.
4. Collects and stores the results in a shared history.

**Do not implement or modify this function.** It is already provided as the orchestration component of the multi-agent pipeline.

Notice that `planner_agent` might return a long list of steps. Because of this, the maximum number of steps is set to a maximum of 4 to keep running time reasonable.

In [ ]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent,
}

def clean_markdown_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:markdown)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [ ]:
class AgentTaskAssignment(BaseModel):
    agent: Literal["research_agent", "editor_agent", "writer_agent"] = Field(description="The agent to which the task is assigned.")
    task: str = Field(description="A string with the instructions that the agent should follow")

In [ ]:
def executor_agent(topic, model: str = "gpt-4o-mini", limit_steps: bool = True):

    plan_steps = planner_agent(topic)

    history = []

    print("==================================")
    print("Executor Agent")
    print("==================================")

    for i, step in enumerate(plan_steps):

        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Instruction: "{step}"
        """
        response = parse_response(
            model=model,
            input=[{"role": "user", "content": agent_decision_prompt}],
            temperature=0,
            text_format=AgentTaskAssignment
        )


        agent_name = response.agent
        task = response.task

        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}" 
            for j, (s, a, r) in enumerate(history)
        ])
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Your next task is:
        {task}
        """

        print(f"\nExecuting with agent: `{agent_name}` on task: {task}")

        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
            history.append((step, agent_name, output))
        else:
            output = f"Unknown agent: {agent_name}"
            history.append((step, agent_name, output))

        print(f"Output:\n{output}")

    return history

In [ ]:
# Keep in mind this could take more than a minute to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting")

md = executor_history[-1][-1]
display(Markdown(clean_markdown_block(md).strip()))